# Stage1 beta0.6 恢复动作：0bp Tiered 训练参考和局部净优势标签

只建立冻结OOF训练参考和标签，不训练新信号或运行2023+候选账户；固定预算与时钟见 plan.json。

In [1]:
import hashlib
import json
import os
import shlex
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.analysis.recovery_training_labels import recovery_training_labels
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec, FEATURE_COLUMNS
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec

os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
root=Path.cwd().resolve();card=root/'research/stage1_signed_volume_0bp_action_label_source_20261023'
plan=json.loads((card/'plan.json').read_text());source=plan['source']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized'] and not (card/'run_state.json').exists()
assert plan['budget']=={'new_market_requests':0,'new_model_fits':0,'new_training_native_accounts':1,'new_label_passes':1,'new_bootstraps':0,'new_Qlib_recorders':1}
for name,digest in plan['code_sha256'].items():assert sha(root/name)==digest,name
assert sha(root/source['training_card']/'plan.json')==source['training_plan_sha256']
assert sha(root/source['provider_uri']/'qlib_store_manifest.json')==source['provider_manifest_sha256']
assert sha(root/source['signed_volume_source_summary'])==source['signed_volume_source_summary_sha256']
signed_source=json.loads((root/source['signed_volume_source_summary']).read_text())
assert signed_source['source_gate_passed'] and signed_source['training_known_days']==1060 and signed_source['held_known_days']==944
for prefix,artifact_digests in [('training',{'inputs/full_reference_frame':'training_frame_artifact_sha256','inputs/selection':'training_selection_artifact_sha256','analysis/native/training_reference':'training_2bp_reference_artifact_sha256'}),('action',{'inputs/plan':'action_plan_artifact_sha256'})]:
    c=root/source[prefix+'_card'];s=json.loads((c/'summary.json').read_text());m=json.loads((c/'verification_manifest.json').read_text());b=json.loads((c/'nas_binding.json').read_text())
    assert sha(c/'verification_manifest.json')==source[prefix+'_manifest_sha256']
    assert s['recorder_id']==source[prefix+'_recorder_id'] and s['nas_evidence']['sha256']==b['evidence']['sha256']==source[prefix+'_NAS_archive_sha256']
    for artifact,key in artifact_digests.items():
        matches=[c/e['path'] for e in m['entries'] if e['path'].endswith('/'+artifact)]
        assert len(matches)==1 and sha(matches[0])==source[key],artifact
qlib.init(provider_uri=str(root/source['provider_uri']),region='us')
def load(prefix,artifact):
    c=root/source[prefix+'_card']
    with R.uri_context(str(c/'qlib_recorder')):
        return R.get_recorder(experiment_name=c.name,recorder_id=source[prefix+'_recorder_id']).load_object(artifact)
frame=load('training','inputs/full_reference_frame')
selection=load('training','inputs/selection')
reference2=load('training','analysis/native/training_reference')
action_plan=load('action','inputs/plan')
assert len(frame)==plan['account']['sessions']==1061
assert str(frame.trade_date.iloc[0].date())==plan['account']['start_trade'] and str(frame.trade_date.iloc[-1].date())==plan['account']['last_trade']
assert frame.trade_date.equals(reference2['authority'].trade_date)
assert len(FEATURE_COLUMNS)==21 and selection['action_beta']==0.6
old=json.loads((root/source['training_card']/'plan.json').read_text())
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action_plan['recovery_spec'])
action=Stage1OpenActionValueSpec(root=root,**action_plan['action_spec'])
no_recovery={'frozen_open_action_value_model':None,'frozen_recovery_model':None}
state={'status':'training_native_account_attempted','plan_sha256':sha(card/'plan.json'),'native_account_attempts':1,'new_model_fits':0,'label_pass_attempts':0}
atomic_json(state,card/'run_state.json')
path=replay_full_path(frame,models=None,plan=old,slippage_bps=0,recovery_spec=recovery,action_spec=action,selection=selection,controller=no_recovery)
assert len(path['replay'])==1061 and not path['replay'].recovery_action.any() and path['replay'].execute_open_target.all()
book=native_tiered_stage1(frame,path,provider_uri=root/source['provider_uri'],tariff=old['tariff'],slippage_bps=0,initial_cash=plan['account']['initial_cash_usd'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition)
assert book['audit']['status']=='passed' and book['audit']['sessions']==1061 and book['audit']['slippage_bps']==0
book['targets']=path['targets']
state.update(status='native_complete',native_accounts_completed=1,label_pass_attempts=1)
atomic_json(state,card/'run_state.json')
labels=recovery_training_labels(frame,book,plan=old,recovery_spec=recovery,action_spec=action,selection=selection,slippage_bps=0)
assert len(labels)==1061 and labels.trade_date.equals(frame.trade_date)
assert labels.loc[labels.net_advantage.notna(),'eligible'].all()
assert labels.loc[~labels.eligible,'net_advantage'].isna().all()
valid=labels.eligible & labels.net_advantage.notna()
assert (labels.loc[valid,'label_available_at']>labels.loc[valid,'decision_at']).all()
annual=[]
for year,g in labels.groupby(labels.trade_date.dt.year,sort=True):
    mature=g.net_advantage.notna()
    annual.append({'year':int(year),'calendar_rows':len(g),'eligible_rows':int(g.eligible.sum()),'mature_eligible_rows':int(mature.sum()),'positive_net_advantage_rows':int(g.loc[mature,'net_advantage'].gt(0).sum()),'negative_net_advantage_rows':int(g.loc[mature,'net_advantage'].lt(0).sum()),'zero_net_advantage_rows':int(g.loc[mature,'net_advantage'].eq(0).sum())})
summary={'experiment':plan['experiment'],'decision':'0bp_Tiered_same_state_recovery_label_source_ready_no_fit','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'training_reference_sessions':len(frame),'native_audit':book['audit'],'label_calendar_rows':len(labels),'eligible_rows':int(labels.eligible.sum()),'mature_eligible_rows':int(valid.sum()),'annual_label_support':annual,'label_last_unknown':str(labels.loc[labels.eligible & labels.net_advantage.isna(),'trade_date'].max().date()) if (labels.eligible & labels.net_advantage.isna()).any() else None,'new_market_requests':0,'new_model_fits':0,'new_native_accounts':1,'new_label_passes':1,'new_bootstraps':0,'historical_independence':False,'held_2023_2026_outcomes_read':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','paths/*','native/*','analysis/labels']},'limitations':['2018-2022 OOF training-reference architecture is not the frozen 2023+ f696 held account; final economic test must use complete Stage1 own-state chain.','Native training reference validates raw integer shares, cash and Tiered fees but does not certify actual broker fills or corporate-action parity.','One-next-session common reset advantage is a local action label, not full policy wealth or independent validation.']}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
    rec=R.get_recorder()
    rec.save_objects(artifact_path='inputs',plan=plan,source_frame=frame,selection=selection,action_plan=action_plan)
    rec.save_objects(artifact_path='paths',reference_0=path)
    rec.save_objects(artifact_path='native',reference_0=book)
    rec.save_objects(artifact_path='analysis',labels=labels,annual_label_support=annual)
    rec.log_metrics(new_model_fits=0,new_native_accounts=1,mature_eligible_rows=int(valid.sum()))
    summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(card/'qlib_recorder'))
atomic_json(summary,card/'summary.json')
binding=json.loads((root/source['training_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2022-12-30')
atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(summary,card/'summary.json')
state.update(status='completed',label_passes_completed=1,recorder_id=summary['recorder_id'])
atomic_json(state,card/'run_state.json')
(card/'conclusion.md').write_text(
    '# Stage1 同状态恢复标签：0bp 训练参考来源\n\n'
    f'冻结2018–2022 OOF原39项配置目标，使用10万美元、原始整数股、0bp额外滑点与仍收取的IBKR Tiered费完成{len(frame)}日连续训练参考账户；无开盘学习型动作和无恢复动作，仅为动作标签的共同起点。Qlib原生现金/股数/费用/NAV对账`{book["audit"]["status"]}`，没有新拟合。\n\n'
    f'在同一开盘后真实数量/现金状态，按原beta0.6和alpha门于10:32分出恢复与HOLD，次日09:47共同目标重置并计入两支Tiered费用，标签09:49后才成熟。合格{summary["eligible_rows"]}日、成熟{summary["mature_eligible_rows"]}日，年度/正负零计数见summary.json，完整逐日标签只在Recorder。原2bp标签没有改名或作为0bp结果复用。\n\n'
    f'决定`{summary["decision"]}`。该参考是复用历史的OOF训练支架，不是2023+原Stage1完整链账户；局部动作优势也不能相加当SOXX超额。下一步要在冻结的训练/评价年份与旧21项输入对照下检验新首小时签名量，失败即关闭。最终仍需原Stage1完整0bp账户、2/5bp独立成本路径及合格前向。Recorder `{summary["recorder_id"]}`，NAS SHA `{remote}`。\n'
)
print(summary['decision'],summary['eligible_rows'],summary['mature_eligible_rows'],annual,summary['recorder_id'])


0bp_Tiered_same_state_recovery_label_source_ready_no_fit 439 438 [{'year': 2018, 'calendar_rows': 53, 'eligible_rows': 9, 'mature_eligible_rows': 9, 'positive_net_advantage_rows': 4, 'negative_net_advantage_rows': 5, 'zero_net_advantage_rows': 0}, {'year': 2019, 'calendar_rows': 252, 'eligible_rows': 19, 'mature_eligible_rows': 19, 'positive_net_advantage_rows': 10, 'negative_net_advantage_rows': 9, 'zero_net_advantage_rows': 0}, {'year': 2020, 'calendar_rows': 253, 'eligible_rows': 145, 'mature_eligible_rows': 145, 'positive_net_advantage_rows': 89, 'negative_net_advantage_rows': 56, 'zero_net_advantage_rows': 0}, {'year': 2021, 'calendar_rows': 252, 'eligible_rows': 27, 'mature_eligible_rows': 27, 'positive_net_advantage_rows': 14, 'negative_net_advantage_rows': 13, 'zero_net_advantage_rows': 0}, {'year': 2022, 'calendar_rows': 251, 'eligible_rows': 239, 'mature_eligible_rows': 238, 'positive_net_advantage_rows': 108, 'negative_net_advantage_rows': 130, 'zero_net_advantage_rows': 0}]

## 中文结论

完成1本2018–2022原Stage1架构OOF训练参考的0bp额外滑点、Tiered费用原生账户和438个成熟beta0.6净恢复/HOLD局部标签。0新拟合；2022-12-30标签因缺下一共同复位保持未知。2018–2020有173个成熟资格、2021–2022有265个，但2021仅27个；不能视为均衡独立年份。原生Qlib核账与NAS 11工件限定恢复通过。此训练标签不是2023+完整Stage1账户或SOXX超额结论。